# EOD Deliverable — Output 1: Plantation Boundary & Palm Validation
## Python / Earth Engine API translation of `21A_MASTER_PIPELINE_OUTPUT1_PLANTATION_BOUNDARY_V1_2_FINAL.js`

**Purpose:** reproducible Python notebook equivalent of the validated GEE JavaScript master pipeline for **SIAK_AOI01, 2025**.

### Parity contract
This notebook preserves the source decisions:
- Sentinel-2 SR Harmonized + Sentinel-1 GRD optical/radar feature stack.
- 20 features: 10 S2 bands + NDVI/NDMI/NDRE/EVI + ASC/DESC VV/VH/ratio.
- 5 internal land-cover classes; final web classes = Non-Palm / Palm / Ambiguous.
- Random Forest: 500 trees, min leaf 2, bag fraction 0.7, seed 42.
- Confidence threshold = 0.60.
- Palm extent smoothing + **MMU 0.50 ha (~50 pixels at 10 m)**.
- Output 1 boundary is the downstream spatial mask; it is **not a legal concession boundary**.

### Operational adaptation from Code Editor to Python
The scientific logic is unchanged. GEE `Export.*` statements are represented by `ee.batch.Export.*` tasks. Tasks are **created but not started automatically**; start them only after QA with `start_tasks(EXPORT_TASKS)`.

**Source SHA-256:** `6f9f75f9e69dcd05b121e780663fe75b6b9e90ec8ff7bf7707490353d6f145fc`

Historical project checkpoint: master pipeline finished; Output 1 boundary used as the primary spatial mask for downstream outputs. Reference project history records **136 boundary polygons** in the finalized AOI01 delivery; area comparisons must state whether they use vector geometry or rasterized area/resolution.


## 0. Environment & Earth Engine authentication

In [ ]:
# OPTIONAL (only if packages are missing):
# %pip install -q earthengine-api folium certifi

import math
import json
import time
from pprint import pprint

try:
    import ssl, certifi
    ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())
except Exception:
    pass

import ee

EE_PROJECT = "palm-oil-503202"

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    # Normal first-run Earth Engine authentication flow.
    # Do NOT paste OAuth tokens into this notebook manually.
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialized")
print("earthengine-api version:", getattr(ee, "__version__", "unknown"))
print("Project:", EE_PROJECT)


In [ ]:
def server_value(obj, label=None):
    """Fetch a small EE object for console-style QA. Avoid on huge rasters/vectors."""
    value = obj.getInfo()
    if label:
        print(f"{label}: {value}")
    else:
        pprint(value)
    return value


def task_summary(tasks):
    """Show batch export tasks created by this notebook."""
    if not tasks:
        print("No export tasks were created.")
        return
    for i, task in enumerate(tasks, 1):
        st = task.status()
        print(f"{i:02d}. {st.get('description')} | state={st.get('state')} | id={task.id}")


def start_tasks(tasks):
    """Start explicitly-created EE batch tasks. Call only after QA."""
    for task in tasks:
        task.start()
        print("Started:", task.status().get("description"), "|", task.id)


## 1. Configuration — source-equivalent values

In [ ]:
CFG = {
    "RUN_MODE": "FINAL",  # source default; change to CHECK for lightweight rerun
    "AOI_ID": "SIAK_AOI01",
    "YEAR": 2025,
    "START_DATE": "2025-01-01",
    "END_DATE": "2026-01-01",
    "CRS": "EPSG:32647",
    "SCALE": 10,
    "AOI_ASSET": "projects/palm-oil-503202/assets/AOI_SIAK_15KM_01_WGS84",
    "SAMPLE_ASSET": "projects/palm-oil-503202/assets/SIAK_AOI01_PILOT_SAMPLE_WGS84_FIXED",
    "EXISTING_STACK_ASSET": "projects/palm-oil-503202/assets/SIAK_AOI01_FEATURE_STACK_OPTIK_RADAR_20B_2025",
    "OUTPUT_ROOT": "projects/palm-oil-503202/assets/",
    "DRIVE_FOLDER": "OUTPUT1_MASTER_SIAK_AOI01_2025_V1_2_FINAL",
    "USE_EXISTING_STACK": True,
    "S2_MAX_SCENE_CLOUD": 60,
    "RF_TREES": 500,
    "RF_MIN_LEAF": 2,
    "RF_BAG_FRACTION": 0.7,
    "RF_SEED": 42,
    "MAX_PIXEL_PER_POLYGON": 200,
    "MAX_PIXEL_PER_CLASS": 600,
    "CONFIDENCE_THRESHOLD": 0.60,
    "MMU_HA": 0.50,
    "SMOOTHING_RADIUS_PIXEL": 1,
    "CONNECTED_MAX_SIZE": 100,
    "EXPORT_CLASSIFIER_ASSET": True,
    "EXPORT_FINAL_RASTER_ASSET": True,
    "EXPORT_BOUNDARY_ASSET": True,
    "EXPORT_SUMMARY_TO_DRIVE": True,
    "EXPORT_FEATURE_STACK_ASSET": False,
    "EXPORT_WEB_DELIVERY": False,
}

IS_CHECK = CFG["RUN_MODE"] == "CHECK"
IS_FINAL = CFG["RUN_MODE"] == "FINAL"
if not (IS_CHECK or IS_FINAL):
    raise ValueError("CFG['RUN_MODE'] must be 'CHECK' or 'FINAL'.")

BUILD_VECTOR_BOUNDARY = IS_FINAL
RUN_FINAL_THREE_CLASS_STATS = IS_FINAL
RUN_AREA_QC = IS_FINAL

# Python safety switches: source logic is preserved, but no export starts by itself.
START_EXPORT_TASKS = False
EAGER_HEAVY_QC = False  # True may force vectorization/large reductions immediately.

BANDS = [
    "B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12",
    "NDVI","NDMI","NDRE","EVI",
    "VV_ASC","VH_ASC","RATIO_ASC",
    "VV_DESC","VH_DESC","RATIO_DESC",
]

print("Run mode:", CFG["RUN_MODE"])
print("Feature count:", len(BANDS))


## 2. Input check + Sentinel-2/Sentinel-1 feature stack

In [ ]:
aoiFC = ee.FeatureCollection(CFG["AOI_ASSET"])
aoi = aoiFC.geometry()
samples = ee.FeatureCollection(CFG["SAMPLE_ASSET"])

print("AOI:", CFG["AOI_ASSET"])
print("Samples:", CFG["SAMPLE_ASSET"])
server_value(aoiFC.size(), "AOI feature count")
server_value(samples.size(), "Sample polygon count")
server_value(ee.Feature(samples.first()).propertyNames(), "Sample properties")
server_value(samples.aggregate_histogram("CLASS_NM"), "Sample classes")
server_value(samples.aggregate_histogram("SPLIT"), "Sample split")

s2_bands = ["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]

def mask_s2(img):
    scl = img.select("SCL")
    mask = (scl.neq(3)
            .And(scl.neq(8))
            .And(scl.neq(9))
            .And(scl.neq(10))
            .And(scl.neq(11)))
    return (img.updateMask(mask)
            .select(s2_bands)
            .multiply(0.0001)
            .copyProperties(img, ["system:time_start"]))


def add_indices(img):
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndmi = img.normalizedDifference(["B8", "B11"]).rename("NDMI")
    ndre = img.normalizedDifference(["B8A", "B5"]).rename("NDRE")
    evi = img.expression(
        "2.5*((nir-red)/(nir+6*red-7.5*blue+1))",
        {"nir": img.select("B8"), "red": img.select("B4"), "blue": img.select("B2")},
    ).rename("EVI")
    return img.addBands([ndvi, ndmi, ndre, evi])

s2Collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi)
    .filterDate(CFG["START_DATE"], CFG["END_DATE"])
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CFG["S2_MAX_SCENE_CLOUD"]))
    .map(mask_s2))

s2Composite = add_indices(s2Collection.median().resample("bilinear").clip(aoi))

s1Base = (ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(aoi)
    .filterDate(CFG["START_DATE"], CFG["END_DATE"])
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    .select(["VV", "VH"]))


def s1_orbit(orbit, suffix):
    med = s1Base.filter(ee.Filter.eq("orbitProperties_pass", orbit)).median().clip(aoi)
    vv = med.select("VV").rename(f"VV_{suffix}")
    vh = med.select("VH").rename(f"VH_{suffix}")
    ratio = vv.subtract(vh).rename(f"RATIO_{suffix}")
    return ee.Image.cat([vv, vh, ratio])

rawStack = (ee.Image.cat([
    s2Composite.select([
        "B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12",
        "NDVI","NDMI","NDRE","EVI",
    ]),
    s1_orbit("ASCENDING", "ASC"),
    s1_orbit("DESCENDING", "DESC"),
]).select(BANDS).clip(aoi).set({
    "AOI_ID": CFG["AOI_ID"],
    "YEAR": CFG["YEAR"],
    "STACK_VERSION": "OUTPUT1_MASTER_RAW_V1_2",
    "FEATURE_COUNT": len(BANDS),
}))

featureStack = (ee.Image(CFG["EXISTING_STACK_ASSET"]).select(BANDS).clip(aoi)
                if CFG["USE_EXISTING_STACK"] else rawStack)

print("Feature stack mode:", "EXISTING VALIDATED STACK" if CFG["USE_EXISTING_STACK"] else "RAW SENTINEL REBUILD")
server_value(s2Collection.size(), "S2 scene count")
server_value(s1Base.filter(ee.Filter.eq("orbitProperties_pass", "ASCENDING")).size(), "S1 ASC scene count")
server_value(s1Base.filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING")).size(), "S1 DESC scene count")
server_value(featureStack.bandNames(), "Feature stack bands")


## 3. Training/validation sampling + Random Forest

In [ ]:
def set_model_label(f):
    f = ee.Feature(f)
    return f.set("LABEL", ee.Number(f.get("CLASS_ID")).subtract(1))

samplesModel = samples.map(set_model_label)
trainPolygons = samplesModel.filter(ee.Filter.eq("SPLIT", "TRAIN"))
validPolygons = samplesModel.filter(ee.Filter.eq("SPLIT", "VALID"))
props = ["SAMP_ID", "CLASS_ID", "CLASS_NM", "SPLIT", "LABEL"]

trainRaw = featureStack.sampleRegions(
    collection=trainPolygons,
    properties=props,
    scale=CFG["SCALE"],
    geometries=False,
    tileScale=4,
)

trainIds = ee.List(trainPolygons.aggregate_array("SAMP_ID")).distinct().sort()

def cap_polygon(id_):
    return (trainRaw
            .filter(ee.Filter.eq("SAMP_ID", id_))
            .randomColumn("RAND_POLYGON", CFG["RF_SEED"])
            .sort("RAND_POLYGON")
            .limit(CFG["MAX_PIXEL_PER_POLYGON"]))

trainCapped = ee.FeatureCollection(trainIds.map(cap_polygon)).flatten()

def cap_class(c):
    return (trainCapped
            .filter(ee.Filter.eq("LABEL", c))
            .randomColumn("RAND_CLASS", CFG["RF_SEED"] + 100)
            .sort("RAND_CLASS")
            .limit(CFG["MAX_PIXEL_PER_CLASS"]))

trainBalanced = ee.FeatureCollection(ee.List.sequence(0, 4).map(cap_class)).flatten()

validPixels = featureStack.sampleRegions(
    collection=validPolygons,
    properties=props,
    scale=CFG["SCALE"],
    geometries=False,
    tileScale=4,
)

server_value(trainPolygons.size(), "TRAIN polygons")
server_value(validPolygons.size(), "VALID polygons")
server_value(trainRaw.size(), "TRAIN pixels raw")
server_value(trainCapped.size(), "TRAIN pixels after polygon cap")
server_value(trainBalanced.size(), "TRAIN balanced pixels")
server_value(trainBalanced.aggregate_histogram("CLASS_NM"), "TRAIN balanced per class")
server_value(validPixels.size(), "VALID pixel count")

rf = ee.Classifier.smileRandomForest(
    numberOfTrees=CFG["RF_TREES"],
    variablesPerSplit=None,
    minLeafPopulation=CFG["RF_MIN_LEAF"],
    bagFraction=CFG["RF_BAG_FRACTION"],
    maxNodes=None,
    seed=CFG["RF_SEED"],
).train(
    features=trainBalanced,
    classProperty="LABEL",
    inputProperties=BANDS,
)

validPred = validPixels.classify(rf, "PREDICTED")
matrix = validPred.errorMatrix("LABEL", "PREDICTED", [0, 1, 2, 3, 4])

print("=== 21A VALIDATION ===")
# rf.explain() can be large; fetch only when needed.
server_value(matrix.array(), "Confusion matrix")
server_value(matrix.accuracy(), "Overall accuracy")
server_value(matrix.kappa(), "Kappa")
server_value(matrix.producersAccuracy(), "Producer accuracy / recall")
server_value(matrix.consumersAccuracy(), "Consumer accuracy / precision")


## 4. Five internal classes → Palm / Non-Palm / Ambiguous

In [ ]:
class5 = featureStack.classify(rf).add(1).toByte().rename("CLASS_ID").clip(aoi)

probArray = featureStack.classify(rf.setOutputMode("MULTIPROBABILITY"))
palmProbability = probArray.arrayGet([0]).rename("PALM_PROBABILITY").clip(aoi)
confidence = (probArray.arrayReduce(ee.Reducer.max(), [0])
              .arrayGet([0]).rename("CLASSIFICATION_CONFIDENCE").clip(aoi))

finalClass = (ee.Image.constant(2)
    .where(confidence.gte(CFG["CONFIDENCE_THRESHOLD"]).And(class5.eq(1)), 1)
    .where(confidence.gte(CFG["CONFIDENCE_THRESHOLD"]).And(class5.neq(1)), 0)
    .rename("PALM_CLASS").toByte().clip(aoi))
# 0 = Non-Palm; 1 = Palm; 2 = Ambiguous

class5Names = ee.Dictionary({
    "1":"PALM", "2":"FOREST_OTHER_TREES", "3":"OPEN_BARE", "4":"BUILT_UP", "5":"WATER"
})
finalNames = ee.Dictionary({"0":"NON_PALM", "1":"PALM", "2":"AMBIGUOUS"})


def area_table(img, code_names, table_name):
    grouped = (ee.Image.pixelArea().divide(10000).rename("AREA_HA")
               .addBands(img.rename("CLASS_CODE"))
               .reduceRegion(
                   reducer=ee.Reducer.sum().group(groupField=1, groupName="CLASS_CODE"),
                   geometry=aoi,
                   scale=CFG["SCALE"],
                   crs=CFG["CRS"],
                   maxPixels=1e13,
                   tileScale=4,
               ))
    groups = ee.List(ee.Dictionary(grouped).get("groups", ee.List([])))

    def make_row(x):
        x = ee.Dictionary(x)
        code = ee.Number(x.get("CLASS_CODE")).toInt()
        return ee.Feature(None, {
            "AOI_ID": CFG["AOI_ID"], "YEAR": CFG["YEAR"], "TABLE_NAME": table_name,
            "CLASS_CODE": code, "CLASS_NAME": code_names.get(code.format("%d")),
            "AREA_HA": ee.Number(x.get("sum")),
        })

    raw = ee.FeatureCollection(groups.map(make_row))
    total = ee.Number(raw.aggregate_sum("AREA_HA"))
    return raw.map(lambda f: ee.Feature(f).set({
        "PERCENT": ee.Number(ee.Feature(f).get("AREA_HA")).divide(total).multiply(100),
        "TOTAL_CLASSIFIED_HA": total,
    })).sort("CLASS_CODE")

class5Stats = area_table(class5, class5Names, "FIVE_CLASS")
finalStats = area_table(finalClass, finalNames, "FINAL_THREE_CLASS") if RUN_FINAL_THREE_CLASS_STATS else ee.FeatureCollection([])

if IS_CHECK:
    server_value(class5Stats, "Five-class statistics")
    print("Final three-class statistics: SKIPPED in CHECK mode")
else:
    print("Area statistics prepared server-side. Fetch/export in FINAL mode as needed.")


## 5. Palm smoothing + MMU 0.50 ha + vector boundary

In [ ]:
minPixels = math.ceil(CFG["MMU_HA"] * 10000 / (CFG["SCALE"] * CFG["SCALE"]))

palmRaw = finalClass.eq(1).selfMask().rename("PALM_RAW").toByte()
palmSmoothed = (palmRaw.unmask(0)
    .focalMax(radius=CFG["SMOOTHING_RADIUS_PIXEL"], kernelType="square", units="pixels", iterations=1)
    .focalMin(radius=CFG["SMOOTHING_RADIUS_PIXEL"], kernelType="square", units="pixels", iterations=1)
    .eq(1).selfMask().toByte())

connectedMaxSize = min(1024, max(minPixels, CFG["CONNECTED_MAX_SIZE"]))
connected = palmSmoothed.connectedPixelCount(maxSize=connectedMaxSize, eightConnected=True)
palmClean = (palmSmoothed.updateMask(connected.gte(minPixels))
             .rename("PALM_MASK_CLEAN").toByte().clip(aoi))

rawPalmHa = ee.Number(0)
cleanPalmHa = ee.Number(0)
areaQc = ee.Dictionary({"PALM_RAW_HA": None, "PALM_CLEAN_HA": None, "AREA_REMOVED_HA": None})

if RUN_AREA_QC:
    ha = ee.Image.pixelArea().divide(10000)
    areaBands = ee.Image.cat([
        ha.updateMask(palmRaw).rename("PALM_RAW_HA"),
        ha.updateMask(palmClean).rename("PALM_CLEAN_HA"),
    ])
    areaResult = ee.Dictionary(areaBands.reduceRegion(
        reducer=ee.Reducer.sum(), geometry=aoi, scale=CFG["SCALE"], crs=CFG["CRS"],
        maxPixels=1e13, tileScale=16,
    ))
    rawPalmHa = ee.Number(areaResult.get("PALM_RAW_HA", 0))
    cleanPalmHa = ee.Number(areaResult.get("PALM_CLEAN_HA", 0))
    areaQc = ee.Dictionary({
        "PALM_RAW_HA": rawPalmHa,
        "PALM_CLEAN_HA": cleanPalmHa,
        "AREA_REMOVED_HA": rawPalmHa.subtract(cleanPalmHa),
    })

print("Confidence threshold:", CFG["CONFIDENCE_THRESHOLD"])
print("MMU (ha):", CFG["MMU_HA"], "| minimum pixels:", minPixels)
print("Connected max size:", connectedMaxSize)
if EAGER_HEAVY_QC and RUN_AREA_QC:
    server_value(areaQc, "Boundary area QC")
else:
    print("Boundary area QC kept lazy (set EAGER_HEAVY_QC=True to fetch).")

boundary = ee.FeatureCollection([])
if BUILD_VECTOR_BOUNDARY:
    boundary = (palmClean.rename("PALM_ID")
        .reduceToVectors(
            reducer=ee.Reducer.countEvery(), geometry=aoi, scale=CFG["SCALE"], crs=CFG["CRS"],
            geometryType="polygon", eightConnected=True, labelProperty="PALM_ID",
            maxPixels=1e13, tileScale=8,
        )
        .map(lambda f: ee.Feature(f).set({
            "AOI_ID": CFG["AOI_ID"], "CLASS_ID": 1, "CLASS_NAME": "PALM", "YEAR": CFG["YEAR"],
            "CONFIDENCE_THRESHOLD": CFG["CONFIDENCE_THRESHOLD"], "MMU_HA": CFG["MMU_HA"],
            "AREA_HA": ee.Feature(f).geometry().area(1).divide(10000),
            "PIPELINE_VERSION": "OUTPUT1_MASTER_V1_2_FINAL",
        }))
        .filter(ee.Filter.gte("AREA_HA", CFG["MMU_HA"])))
    print("Vector boundary prepared server-side. Do not force rendering unless needed.")


## 6. Final stack, summary package, metadata

In [ ]:
finalStack = ee.Image.cat([
    class5, finalClass, palmProbability, confidence,
    palmRaw.unmask(0), palmClean.unmask(0),
    connected.rename("CONNECTED_PIXEL_COUNT").toInt16(),
]).clip(aoi).set({
    "AOI_ID": CFG["AOI_ID"], "OUTPUT_ID": "OUTPUT1_PLANTATION_BOUNDARY", "YEAR": CFG["YEAR"],
    "MODEL": "Random Forest", "CONFIDENCE_THRESHOLD": CFG["CONFIDENCE_THRESHOLD"],
    "MMU_HA": CFG["MMU_HA"], "PIPELINE_VERSION": "OUTPUT1_MASTER_V1_2_FINAL",
})

validationSummary = ee.FeatureCollection([ee.Feature(None, {
    "AOI_ID": CFG["AOI_ID"], "YEAR": CFG["YEAR"],
    "STACK_MODE": "EXISTING_VALIDATED_STACK" if CFG["USE_EXISTING_STACK"] else "RAW_SENTINEL_REBUILD",
    "RF_TREES": CFG["RF_TREES"], "TRAIN_POLYGONS": trainPolygons.size(),
    "VALID_POLYGONS": validPolygons.size(), "TRAIN_PIXELS": trainBalanced.size(),
    "VALID_PIXELS": validPixels.size(), "OVERALL_ACCURACY": matrix.accuracy(), "KAPPA": matrix.kappa(),
    "CONFIDENCE_THRESHOLD": CFG["CONFIDENCE_THRESHOLD"], "MMU_HA": CFG["MMU_HA"],
    "PALM_RAW_HA": rawPalmHa, "PALM_CLEAN_HA": cleanPalmHa,
})])

metadata = ee.FeatureCollection([ee.Feature(None, {
    "PRODUCT": "Plantation Boundary and Palm Validation", "AOI_ID": CFG["AOI_ID"], "YEAR": CFG["YEAR"],
    "SCALE_M": CFG["SCALE"], "CRS": CFG["CRS"], "MODEL": "Random Forest",
    "INTERNAL_CLASSES": "1 Palm; 2 Forest/Other Trees; 3 Open/Bare; 4 Built-up; 5 Water",
    "FINAL_CLASSES": "0 Non-Palm; 1 Palm; 2 Ambiguous",
    "CONFIDENCE_THRESHOLD": CFG["CONFIDENCE_THRESHOLD"], "MMU_HA": CFG["MMU_HA"],
    "LIMITATION": "Model-derived plantation extent; not a legal concession boundary.",
})])

boundarySummary = ee.FeatureCollection([ee.Feature(None, {
    "RECORD_TYPE": "BOUNDARY_QC", "AOI_ID": CFG["AOI_ID"], "YEAR": CFG["YEAR"],
    "CONFIDENCE_THRESHOLD": CFG["CONFIDENCE_THRESHOLD"], "MMU_HA": CFG["MMU_HA"],
    "MIN_PIXELS": minPixels, "PALM_RAW_HA": rawPalmHa, "PALM_CLEAN_HA": cleanPalmHa,
    "AREA_REMOVED_HA": rawPalmHa.subtract(cleanPalmHa),
    "BOUNDARY_POLYGON_COUNT": boundary.size(),
    "PIPELINE_VERSION": "OUTPUT1_MASTER_V1_2_FINAL",
})])

summaryPackage = (class5Stats.map(lambda f: ee.Feature(f).set("RECORD_TYPE", "FIVE_CLASS_STATS"))
    .merge(finalStats.map(lambda f: ee.Feature(f).set("RECORD_TYPE", "FINAL_THREE_CLASS_STATS")))
    .merge(validationSummary.map(lambda f: ee.Feature(f).set("RECORD_TYPE", "VALIDATION_SUMMARY")))
    .merge(boundarySummary)
    .merge(metadata.map(lambda f: ee.Feature(f).set("RECORD_TYPE", "METADATA"))))

server_value(finalStack.bandNames(), "Final stack bands")
print("Summary package prepared.")


## 7. Optional visualization (Folium, no geemap required)

In [ ]:
def make_ee_folium_map(center, zoom=11):
    """Minimal Earth Engine map viewer without geemap."""
    import folium

    m = folium.Map(location=center, zoom_start=zoom, control_scale=True)

    def add_ee_image(image, vis, name, shown=True, opacity=1.0):
        map_id = ee.Image(image).getMapId(vis)
        folium.TileLayer(
            tiles=map_id["tile_fetcher"].url_format,
            attr="Google Earth Engine",
            name=name,
            overlay=True,
            control=True,
            show=shown,
            opacity=opacity,
        ).add_to(m)

    return m, add_ee_image


In [ ]:
center_xy = aoi.centroid(1).coordinates().getInfo()
center = [center_xy[1], center_xy[0]]
Map, add_ee_image = make_ee_folium_map(center, zoom=11)

add_ee_image(featureStack.select(["B4","B3","B2"]), {"min":0.02,"max":0.30,"gamma":1.2}, "RGB 2025", shown=False)
add_ee_image(class5, {"min":1,"max":5,"palette":["006400","7CB342","D9B38C","D32F2F","1976D2"]}, "RF 5 Classes", shown=False)
add_ee_image(finalClass, {"min":0,"max":2,"palette":["BDBDBD","006400","FF9800"]}, "Palm / Non-Palm / Ambiguous", shown=False)
add_ee_image(palmClean, {"min":1,"max":1,"palette":["FFFF00"]}, "Palm Mask after MMU", shown=True)
add_ee_image(samples.style(color="FFFFFF", fillColor="00000000", width=2), {}, "Samples", shown=False)

import folium
folium.LayerControl(collapsed=False).add_to(Map)
Map


## 8. Batch exports — source-equivalent, safe by default

In [ ]:
EXPORT_TASKS = []
base = f"{CFG['AOI_ID']}_OUTPUT1_PLANTATION_BOUNDARY_{CFG['YEAR']}_V1_2_FINAL"

if IS_FINAL and CFG["EXPORT_FEATURE_STACK_ASSET"]:
    EXPORT_TASKS.append(ee.batch.Export.image.toAsset(
        image=rawStack.toFloat(), description=base + "_FEATURE_STACK_ASSET",
        assetId=CFG["OUTPUT_ROOT"] + base + "_FEATURE_STACK_20B",
        region=aoi, scale=CFG["SCALE"], crs=CFG["CRS"], maxPixels=1e13,
    ))

if IS_FINAL and CFG["EXPORT_CLASSIFIER_ASSET"]:
    EXPORT_TASKS.append(ee.batch.Export.classifier.toAsset(
        classifier=rf, description=base + "_RF_CLASSIFIER_ASSET",
        assetId=CFG["OUTPUT_ROOT"] + base + "_RF_CLASSIFIER",
    ))

if IS_FINAL and CFG["EXPORT_FINAL_RASTER_ASSET"]:
    EXPORT_TASKS.append(ee.batch.Export.image.toAsset(
        image=finalStack.toFloat(), description=base + "_FINAL_RASTER_ASSET",
        assetId=CFG["OUTPUT_ROOT"] + base + "_FINAL_RASTER_STACK",
        region=aoi, scale=CFG["SCALE"], crs=CFG["CRS"], maxPixels=1e13,
    ))

if IS_FINAL and CFG["EXPORT_BOUNDARY_ASSET"] and BUILD_VECTOR_BOUNDARY:
    EXPORT_TASKS.append(ee.batch.Export.table.toAsset(
        collection=boundary, description=base + "_BOUNDARY_ASSET",
        assetId=CFG["OUTPUT_ROOT"] + base + "_BOUNDARY",
    ))

if IS_FINAL and CFG["EXPORT_SUMMARY_TO_DRIVE"]:
    EXPORT_TASKS.append(ee.batch.Export.table.toDrive(
        collection=summaryPackage, description=base + "_SUMMARY_PACKAGE_CSV",
        folder=CFG["DRIVE_FOLDER"], fileNamePrefix=base + "_SUMMARY_PACKAGE", fileFormat="CSV",
    ))

if IS_FINAL and CFG["EXPORT_WEB_DELIVERY"]:
    nodataClass, nodataFloat = 255, -9999

    def drive_raster(img, name, nodata, byte=False):
        out = img.unmask(nodata).toUint8() if byte else img.unmask(nodata).toFloat()
        return ee.batch.Export.image.toDrive(
            image=out, description=name, folder=CFG["DRIVE_FOLDER"], fileNamePrefix=name,
            region=aoi, scale=CFG["SCALE"], crs=CFG["CRS"], maxPixels=1e13,
            fileFormat="GeoTIFF", formatOptions={"cloudOptimized": True, "noData": nodata},
        )

    EXPORT_TASKS += [
        drive_raster(class5, f"{CFG['AOI_ID']}_RF_CLASSIFICATION_5CLASS_{CFG['YEAR']}", nodataClass, True),
        drive_raster(finalClass, f"{CFG['AOI_ID']}_FINAL_PALM_NONPALM_AMBIGUOUS_{CFG['YEAR']}", nodataClass, True),
        drive_raster(palmProbability, f"{CFG['AOI_ID']}_PALM_PROBABILITY_{CFG['YEAR']}", nodataFloat),
        drive_raster(confidence, f"{CFG['AOI_ID']}_CLASSIFICATION_CONFIDENCE_{CFG['YEAR']}", nodataFloat),
        drive_raster(palmClean, f"{CFG['AOI_ID']}_PALM_MASK_MMU_{CFG['YEAR']}", 0, True),
        ee.batch.Export.table.toDrive(collection=class5Stats, description=f"{CFG['AOI_ID']}_STATISTICS_5CLASS_{CFG['YEAR']}", folder=CFG["DRIVE_FOLDER"], fileNamePrefix=f"{CFG['AOI_ID']}_STATISTICS_5CLASS_{CFG['YEAR']}", fileFormat="CSV"),
        ee.batch.Export.table.toDrive(collection=finalStats, description=f"{CFG['AOI_ID']}_STATISTICS_FINAL_3CLASS_{CFG['YEAR']}", folder=CFG["DRIVE_FOLDER"], fileNamePrefix=f"{CFG['AOI_ID']}_STATISTICS_FINAL_3CLASS_{CFG['YEAR']}", fileFormat="CSV"),
        ee.batch.Export.table.toDrive(collection=validationSummary, description=f"{CFG['AOI_ID']}_VALIDATION_SUMMARY_{CFG['YEAR']}", folder=CFG["DRIVE_FOLDER"], fileNamePrefix=f"{CFG['AOI_ID']}_VALIDATION_SUMMARY_{CFG['YEAR']}", fileFormat="CSV"),
        ee.batch.Export.table.toDrive(collection=metadata, description=f"{CFG['AOI_ID']}_OUTPUT1_METADATA_{CFG['YEAR']}", folder=CFG["DRIVE_FOLDER"], fileNamePrefix=f"{CFG['AOI_ID']}_OUTPUT1_METADATA_{CFG['YEAR']}", fileFormat="CSV"),
    ]

    if BUILD_VECTOR_BOUNDARY:
        boundary4326 = boundary.map(lambda f: ee.Feature(
            ee.Feature(f).geometry().transform("EPSG:4326", 1), ee.Feature(f).toDictionary()
        ))
        EXPORT_TASKS.append(ee.batch.Export.table.toDrive(
            collection=boundary4326,
            description=f"{CFG['AOI_ID']}_PLANTATION_BOUNDARY_WEB_WGS84_{CFG['YEAR']}",
            folder=CFG["DRIVE_FOLDER"],
            fileNamePrefix=f"{CFG['AOI_ID']}_PLANTATION_BOUNDARY_WEB_WGS84_{CFG['YEAR']}",
            fileFormat="GeoJSON",
        ))

task_summary(EXPORT_TASKS)
if START_EXPORT_TASKS:
    start_tasks(EXPORT_TASKS)
else:
    print("Exports NOT started. After QA, run: start_tasks(EXPORT_TASKS)")


## 9. Final parity checklist
Expected configuration inherited from the final GEE pipeline:
- AOI: `AOI_SIAK_15KM_01_WGS84`
- CLASS_ID: 1–5; training/validation split from the fixed sample asset
- Feature bands: **20**
- RF trees: **500**
- Confidence threshold: **0.60**
- MMU: **0.50 ha = 50 pixels at 10 m**
- Internal 5 classes → final `Non-Palm / Palm / Ambiguous`
- Historical finalized delivery reference: **136 plantation-boundary polygons** (do not use as a hard runtime assertion because vector/raster area methods differ).

For a clean mentor comparison, compare:
1. input counts/bands;
2. confusion matrix, OA, Kappa, producer/consumer accuracy;
3. 5-class and final 3-class area tables;
4. clean palm-mask spatial pattern;
5. exported final raster/boundary asset names and metadata.
